In [1]:
from src.portfolioagent import PortfolioAgent

agent = PortfolioAgent()

response = agent.answer_question(question=input("question"))
response

GenerateContentResponse(
  automatic_function_calling_history=[
    UserContent(
      parts=[
        Part(
          text='What are the sector exposures for the Tech Innovation Fund?'
        ),
      ],
      role='user'
    ),
    Content(
      parts=[
        Part(
          function_call=FunctionCall(
            args={<... 1 item at Max depth ...>},
            name='exposure_calculator'
          ),
          thought_signature=b"\n\x8c\x03\x01\x0c9\xd6\xc7\x08\x1b\x89\x85\xc02\xf1\x9e\xa5V!\x80\xd1\xfeo\xe58\xe2e\xe7\xcc\x9c\x8f\x82\x90Z\x13\x88\x95j\xde\x0f\x87\x14\xa1\xb5*S\\srf\xbb\x83im\x15\xb0\xdc\xa3\xf9\r\xf3\xb4\xafH\xbaWp\x93R\xf7\xa6|\xcb\xe3\x8c'&f\xfd&\xea\xb7\x1f\xc5\xb8J\xd7\xe9\xb8\xaf\x92\r\x0f.\xba\xbf\xa5...'
        ),
      ],
      role='model'
    ),
    Content(
      parts=[
        Part(
          function_response=FunctionResponse(
            name='exposure_calculator',
            response={<... 1 item at Max depth ...>}
          )
        ),
     

In [2]:
def answer(question):
    return agent.answer_question(question=question)

In [3]:
print(response.text)

The sector exposure for the Tech Innovation Fund is 100% in Technology.


In [4]:
response.automatic_function_calling_history

[UserContent(
   parts=[
     Part(
       text='What are the sector exposures for the Tech Innovation Fund?'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'portfolio_name': 'Tech Innovation Fund'
         },
         name='exposure_calculator'
       ),
       thought_signature=b"\n\x8c\x03\x01\x0c9\xd6\xc7\x08\x1b\x89\x85\xc02\xf1\x9e\xa5V!\x80\xd1\xfeo\xe58\xe2e\xe7\xcc\x9c\x8f\x82\x90Z\x13\x88\x95j\xde\x0f\x87\x14\xa1\xb5*S\\srf\xbb\x83im\x15\xb0\xdc\xa3\xf9\r\xf3\xb4\xafH\xbaWp\x93R\xf7\xa6|\xcb\xe3\x8c'&f\xfd&\xea\xb7\x1f\xc5\xb8J\xd7\xe9\xb8\xaf\x92\r\x0f.\xba\xbf\xa5...'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='exposure_calculator',
         response={
           'result': {
             'portfolio': 'Tech Innovation Fund',
             'sector_exposures': [<... 1 item at Max depth ...>]
           }
         }
     

In [5]:
# =========================
# FINAL RESPONSE
# =========================

print("\n=== FINAL RESPONSE ===")
print(response.text)

# =========================
# MODEL INFO
# =========================

print("\n=== MODEL INFO ===")
print("Model:", response.model_version)
print("Response ID:", response.response_id)

# =========================
# TOKEN USAGE
# =========================

usage = response.usage_metadata

print("\n=== TOKEN USAGE ===")
print("Prompt Tokens:", usage.prompt_token_count)
print("Completion Tokens:", usage.candidates_token_count)
print("Total Tokens:", usage.total_token_count)

# =========================
# TOOL CALL BREAKDOWN
# =========================

print("\n=== TOOL CALLS ===")

history = response.automatic_function_calling_history

for item in history:

    # Model calling function
    if item.role == "model":

        for part in item.parts:

            if hasattr(part, "function_call") and part.function_call:

                print("\nTool Used:")
                print("Function:", part.function_call.name)

                print("Arguments:")
                print(part.function_call.args)

    # Function response
    elif item.role == "user":

        for part in item.parts:

            if hasattr(part, "function_response") and part.function_response:

                print("\nTool Response:")
                print(part.function_response.response)


=== FINAL RESPONSE ===
The sector exposure for the Tech Innovation Fund is 100% in Technology.

=== MODEL INFO ===
Model: gemini-2.5-flash
Response ID: m4QAate5C9TojuMPj66e6Qg

=== TOKEN USAGE ===
Prompt Tokens: 169
Completion Tokens: 17
Total Tokens: 186

=== TOOL CALLS ===

Tool Used:
Function: exposure_calculator
Arguments:
{'portfolio_name': 'Tech Innovation Fund'}

Tool Response:
{'result': {'portfolio': 'Tech Innovation Fund', 'sector_exposures': [{'sector_name': 'Technology', 'exposure_percentage': 1.0}]}}


In [ ]:
def extract_tool_results_text2sql(response):
    extracted = []
    current_tool = None

    for item in response.automatic_function_calling_history:

        # Capture tool used
        if item.role == "model":
            for part in item.parts:
                if hasattr(part, "function_call") and part.function_call:
                    current_tool = part.function_call.name

        # Capture tool response
        elif item.role == "user":
            for part in item.parts:

                if hasattr(part, "function_response") and part.function_response:
                    result_data = part.function_response.response.get("result", {})

                    extracted.append({
                        "tool_used": current_tool,
                        "query": result_data.get("sql"),
                        "result": result_data.get("result"),
                        "result_type": result_data.get("type"),
                    })

    return extracted

In [ ]:
extract_tool_results_text2sql(response)

[{'tool_used': 'ask_database',
  'query': 'SELECT COUNT(*) FROM portfolios',
  'result': 13,
  'result_type': 'single_value'}]

In [ ]:
def extract_tool_results_exposure_calculator(response):
    extracted = []
    current_tool = None

    for item in response.automatic_function_calling_history:

        # Capture tool used
        if item.role == "model":
            for part in item.parts:
                if hasattr(part, "function_call") and part.function_call:
                    current_tool = part.function_call.name

        # Capture tool response
        elif item.role == "user":
            for part in item.parts:

                if hasattr(part, "function_response") and part.function_response:
                    result_data = part.function_response.response.get("result", {})

                    extracted.append({
                        "tool_used": current_tool,
                        "portfolio_name": result_data.get("portfolio"),
                        "sector_exposures": result_data.get("sector_exposures")
                    })

    return extracted

In [16]:
res = extract_tool_results_exposure_calculator(response)

In [20]:
res[0]

{'tool_used': 'exposure_calculator',
 'portfolio_name': 'Tech Innovation Fund',
 'sector_exposures': [{'sector_name': 'Technology',
   'exposure_percentage': 1.0}]}

In [23]:
print(res[0]['tool_used'])
print(res[0]['portfolio_name'])
print(res[0]['sector_exposures'])

exposure_calculator
Tech Innovation Fund
[{'sector_name': 'Technology', 'exposure_percentage': 1.0}]


# Evaluation


In [11]:
import json

In [12]:
file_path="ground_truth_dataset.json"
with open(file_path, "r") as f:
    data = json.load(f)

In [ ]:
questions = data['questions']
total = len(questions)
correct = 0

In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq


def groq_sql_judge(query1, query2):
    """
    Uses Groq LLM as a judge to determine whether two SQL queries
    are semantically equivalent.

    Returns:
        bool -> True if equivalent, else False
    """

    # Load environment variables
    load_dotenv()

    api_key = os.getenv("GROQ_API_KEY")

    if not api_key:
        raise ValueError("GROQ_API_KEY not found in .env file")

    client = Groq(api_key=api_key)

    prompt = f"""
You are an expert SQL evaluator.

Your task is to determine whether TWO SQL queries are semantically equivalent.

Rules:
- Ignore formatting differences.
- Ignore alias naming differences.
- Ignore capitalization differences.
- Ignore column order ONLY if result semantics remain same.
- Focus on whether both queries return the SAME logical result.
- Return STRICT JSON only.

Output format:
{{
    "equivalent": true
}}

OR

{{
    "equivalent": false
}}

Query 1:
{query1}

Query 2:
{query2}
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": "You are a strict SQL semantic equivalence judge."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        content = response.choices[0].message.content
        parsed = json.loads(content)

        return bool(parsed.get("equivalent", False))

    except Exception as e:
        print(f"[groq_sql_judge ERROR] {e}")
        return False

In [ ]:
for q in questions:
    import time

    time.sleep(60)
    qid = q["id"]
    question = q["question"]
    qtype = q["type"]
    gt = q["ground_truth"]

    
    print(f"\n[Q{qid}] {question}")

    response = agent.answer_question(question)
    
    if q['type']== 'text2sql':
        expected_sql = gt["sql_query"]
        expected_result_type = gt["expected_result_type"]
        

        generated_data = extract_tool_results_text2sql(response=response)
        query = generated_data[0]['query']
        tool_used = generated_data[0]['tool_used']
        result_type = generated_data[0]['result_type']

        if not query:
            print("❌ No SQL generated by agent")
            continue
        is_correct = groq_sql_judge(expected_sql, query)

        print("\n--- GROQ JUDGE ---")
        print("Match:", is_correct)



        
    
    elif qtype == "exposure_calculator":
        expected_tool = gt["tool_name"]
        
        generated_data = extract_tool_results_exposure_calculator(response)
        tool_used = generated_data[0]['tool_used']

        if expected_tool == tool_used:
            is_correct = True

        else:
            is_correct = False
        
        print("Match:", is_correct)

    if is_correct:
        correct += 1 
        

accuracy = (correct / total) * 100


print("\n================ FINAL RESULTS ================\n")
print(f"Total Questions: {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {accuracy:.2f}%")
print("\n==============================================\n")


